In [1]:
DATA_DICT = {
    "compas_race": {"sens_attrs": ["Race"], "label": "Two_yr_Recidivism", "favored": (0)},
    "compas_sex": {"sens_attrs": ["Female"], "label": "Two_yr_Recidivism", "favored": (0)},
    "communities": {"sens_attrs": ["race"], "label": "crime", "favored": (0)},
    "credit_card_clients": {"sens_attrs": ["sex"], "label": "payment", "favored": (1)},
    "adult_data_set_race": {"sens_attrs": ["race"], "label": "salary", "favored": (0)},
    "adult_data_set_sex": {"sens_attrs": ["sex"], "label": "salary", "favored": (0)},
    "ACS_ID_norm": {"sens_attrs": ["SEX"], "label": "PINCP", "favored": (1)},
    "ACS_OR_norm": {"sens_attrs": ["SEX"], "label": "PINCP", "favored": (1)},
    "drug_consumption": {"sens_attrs": ["gender"], "label": "cannabis", "favored": (1)},
}

In [2]:
import os
import numpy as np
import pandas as pd
import copy
import math
import time

from fractions import Fraction

from aif360.datasets import BinaryLabelDataset
from scipy.stats import pearsonr

from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split

from algorithm.FALCCAddons.parameter_estimation import log_means
from SFC.ClusteringMethod import ScalableClustering
from SFC.SFC import SFC
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids


# TODO: adjust this import to wherever FairletKMeans2 actually lives
from algorithm.FALCC_Alts import FairletKMeans2, FairletConstruction   # or from <your_module> import FairletKMeans2


def eval_kmeans(X_train_cluster, X_test_cluster, sens_train, sens_test, k, random_state=0):
    """
    Classical KMeans (no fairness constraints).

    - Fit on train features
    - Predict clusters for train and test
    - Compute balance + silhouette on both
    """
    km = KMeans(n_clusters=k, random_state=random_state, n_init="auto")
    km.fit(X_train_cluster.to_numpy(dtype=float))

    cluster_train = km.predict(X_train_cluster.to_numpy(dtype=float))
    cluster_test = km.predict(X_test_cluster.to_numpy(dtype=float))

    bal_train = compute_proportional_balance_scores(cluster_train, sens_train)
    bal_test = compute_proportional_balance_scores(cluster_test, sens_test)
    bal_train_diff = compute_balance_diff_scores(cluster_train, sens_train)
    bal_test_diff = compute_balance_diff_scores(cluster_test, sens_test)

    res = {
        "train_balance_worst": bal_train[0],
        "train_balance_mean": bal_train[1],
        "train_bal_diff_worst": bal_train_diff[0],
        "train_bal_diff_mean": bal_train_diff[1],
        "train_silhouette": compute_silhouette_safe(
            X_train_cluster.to_numpy(dtype=float), cluster_train
        ),
        "test_balance_worst": bal_test[0],
        "test_balance_mean": bal_test[1],
        "test_bal_diff_worst": bal_test_diff[0],
        "test_bal_diff_mean": bal_test_diff[1],
        "test_silhouette": compute_silhouette_safe(
            X_test_cluster.to_numpy(dtype=float), cluster_test
        ),
    }
    return res


def _build_result_dict(X_train_np, X_test_np, cluster_train, cluster_test,
                       sens_train, sens_test):
    bal_train = compute_proportional_balance_scores(cluster_train, sens_train)
    bal_test = compute_proportional_balance_scores(cluster_test, sens_test)
    bal_train_diff = compute_balance_diff_scores(cluster_train, sens_train)
    bal_test_diff = compute_balance_diff_scores(cluster_test, sens_test)
 
    return {
        "train_balance_worst": bal_train[0],
        "train_balance_mean": bal_train[1],
        "train_bal_diff_worst": bal_train_diff[0],
        "train_bal_diff_mean": bal_train_diff[1],
        "train_silhouette": compute_silhouette_safe(X_train_np, cluster_train),
        "test_balance_worst": bal_test[0],
        "test_balance_mean": bal_test[1],
        "test_bal_diff_worst": bal_test_diff[0],
        "test_bal_diff_mean": bal_test_diff[1],
        "test_silhouette": compute_silhouette_safe(X_test_np, cluster_test),
    }


def compute_balance_score(cluster_labels, sens_values):
    labels = np.asarray(cluster_labels)
    sens = np.asarray(sens_values)

    unique_clusters = np.unique(labels)
    if len(unique_clusters) <= 1:
        return 0.0

    per_cluster_balance = []
    for c in unique_clusters:
        mask = labels == c
        if not np.any(mask):
            continue
        sens_c = sens[mask]
        counts = np.bincount(sens_c.astype(int))
        if np.any(counts == 0):
            # at least one group missing in this cluster
            per_cluster_balance.append(0.0)
        else:
            per_cluster_balance.append(counts.min() / counts.max())

    if not per_cluster_balance:
        return 0.0
    return float(np.min(per_cluster_balance))


def compute_proportional_balance_scores(cluster_labels, sens_values, eps=1e-12):
    labels = np.asarray(cluster_labels)
    sens = np.asarray(sens_values).astype(int)

    p = sens.mean()  # global fraction of group 1
    unique_clusters = np.unique(labels)

    # Degenerate case: only one group globally
    if p < eps or (1 - p) < eps:
        return 1.0, 1.0

    cluster_scores = []
    for c in unique_clusters:
        mask = labels == c
        if not np.any(mask):
            continue

        pk = sens[mask].mean()

        # representation ratio
        score = min(pk / p, (1 - pk) / (1 - p))

        # numerical safety
        score = max(0.0, min(1.0, float(score)))
        cluster_scores.append(score)

    if not cluster_scores:
        return 0.0, 0.0

    return float(np.min(cluster_scores)), float(np.mean(cluster_scores))


def compute_balance_diff_scores(cluster_labels, sens_values, eps=1e-12):
    labels = np.asarray(cluster_labels)
    sens = np.asarray(sens_values).astype(int)

    p = sens.mean()  # global fraction of group 1
    unique_clusters = np.unique(labels)

    # Degenerate case: only one group globally
    if p < eps or (1 - p) < eps:
        return 1.0, 1.0

    cluster_scores = []
    for c in unique_clusters:
        mask = labels == c
        if not np.any(mask):
            continue

        pk = sens[mask].mean()

        # representation ratio
        score = abs(pk - p)

        cluster_scores.append(score)

    if not cluster_scores:
        return 0.0, 0.0

    return float(np.max(cluster_scores)), float(np.mean(cluster_scores))


def compute_silhouette_safe(X, labels):
    X_arr = np.asarray(X)
    labels_arr = np.asarray(labels)

    if X_arr.shape[0] <= 2:
        return float("nan")
    if len(np.unique(labels_arr)) <= 1:
        return float("nan")

    try:
        return float(silhouette_score(X_arr, labels_arr))
    except Exception:
        return float("nan")


def infer_sfc_pq_small(
    sensitive,
    max_q=10,
    min_balance=None,
):
    s = pd.Series(np.asarray(sensitive).ravel()).dropna()
    counts = s.value_counts()

    if len(counts) != 2:
        raise ValueError(
            f"Expected binary sensitive attribute, got {len(counts)} groups: "
            f"{counts.to_dict()}"
        )

    minority_count = int(counts.min())
    majority_count = int(counts.max())

    global_balance = minority_count / majority_count

    target_balance = global_balance
    if min_balance is not None:
        target_balance = max(target_balance, min_balance)

    target_balance = min(target_balance, 1.0)

    # Approximate p/q using small integers
    frac = Fraction(target_balance).limit_denominator(max_q)

    p = frac.numerator
    q = frac.denominator

    info = {
        "minority_count": minority_count,
        "majority_count": majority_count,
        "global_balance": global_balance,
        "target_balance": target_balance,
        "p": p,
        "q": q,
        "p_over_q": p / q,
        "p_plus_q": p + q,
    }

    return p, q, info
    
def estimate_k_log_means(X_cluster: pd.DataFrame,
                         df_train: pd.DataFrame,
                         df_dict: dict,
                         ccr=(-1, -1)) -> int:
    sens_attrs = df_dict["sens_attrs"]

    if ccr[0] == ccr[1] and ccr[0] != -1:
        return int(ccr[0])

    sens_groups = len(df_train.groupby(sens_attrs))

    if ccr[0] == -1:
        min_cluster = min(
            len(X_cluster.columns),
            int(len(X_cluster) / (50 * sens_groups)),
        )
    else:
        min_cluster = ccr[0]

    if ccr[1] == -1:
        max_cluster = min(
            int(len(X_cluster.columns) ** 2 / 2),
            int(len(X_cluster) / (10 * sens_groups)),
        )
    else:
        max_cluster = ccr[1]

    k = log_means(X_cluster, min_cluster, max_cluster)
    return int(k)


def eval_sfc(X_train_cluster, X_test_cluster, sens_train, sens_test, k, dataset_name):
    """
    SFC: Scalable fair clustering.
    """
    p, q, info = infer_sfc_pq_small(
        sens_train,
        max_q=10,
        min_balance=None,
    )

    
    sfc = SFC(p=p, q=q+1, k=k)
    sfc.fit(X_train_cluster.to_numpy(dtype=float), sens_train.astype(float))

    cluster_train = sfc.predict(X_train_cluster.to_numpy(dtype=float))
    cluster_test = sfc.predict(X_test_cluster.to_numpy(dtype=float))
    bal_train = compute_proportional_balance_scores(cluster_train, sens_train)
    bal_test = compute_proportional_balance_scores(cluster_test, sens_test)
    bal_train_diff = compute_balance_diff_scores(cluster_train, sens_train)
    bal_test_diff = compute_balance_diff_scores(cluster_test, sens_test)

    res = {
        "train_balance_worst": bal_train[0],
        "train_balance_mean": bal_train[1],
        "train_bal_diff_worst": bal_train_diff[0],
        "train_bal_diff_mean": bal_train_diff[1],
        "train_silhouette": compute_silhouette_safe(
            X_train_cluster.to_numpy(dtype=float), cluster_train
        ),
        "test_balance_worst": bal_test[0],
        "test_balance_mean": bal_test[1],
        "test_bal_diff_worst": bal_test_diff[0],
        "test_bal_diff_mean": bal_test_diff[1],
        "test_silhouette": compute_silhouette_safe(
            X_test_cluster.to_numpy(dtype=float), cluster_test
        ),
    }
    return res
    

def eval_fairlet(X_train_cluster, X_test_cluster, sens_train, sens_test, mode, k, bal="-", cluster_strat=("kmeans", "classic"), random_state=42):
    """
    FairletKMeans2: builds fairlets, then clusters fairlet centroids via KMeans.

    Note: FairletKMeans2 currently chooses its own #clusters internally via log_means,
    so k might differ from SFC/PFC/FCA. If you want matching k, extend FairletKMeans2
    with an n_clusters parameter.
    """
    # Binary / categorical codes for sensitive attribute
    sens_codes, _ = pd.factorize(sens_train)

    if mode == "fairlet":
        fairlet_km = FairletKMeans2(random_state=random_state)
        fairlet_km.fit(
            X_train_cluster.to_numpy(dtype=float),
            sens_codes,
        )

    else:
        fairlet_km = FairletConstruction(mode=mode, n_clusters=k, cluster_strat=cluster_strat, random_state=42, bal=bal)
        fairlet_km.fit(
            X_train_cluster.to_numpy(dtype=float),
            sens_codes,
        )
        
    # Predict clusters on unseen test data
    cluster_train = fairlet_km.predict(X_train_cluster.to_numpy(dtype=float))
    cluster_test = fairlet_km.predict(X_test_cluster.to_numpy(dtype=float))

    # Use the internal clustersize for logging if desired
    k_used = fairlet_km.return_clustersize()

    bal_train = compute_proportional_balance_scores(cluster_train, sens_train)
    bal_test = compute_proportional_balance_scores(cluster_test, sens_test)
    bal_train_diff = compute_balance_diff_scores(cluster_train, sens_train)
    bal_test_diff = compute_balance_diff_scores(cluster_test, sens_test)

    res = {
        "train_balance_worst": bal_train[0],
        "train_balance_mean": bal_train[1],
        "train_bal_diff_worst": bal_train_diff[0],
        "train_bal_diff_mean": bal_train_diff[1],
        "train_silhouette": compute_silhouette_safe(
            X_train_cluster.to_numpy(dtype=float), cluster_train
        ),
        "test_balance_worst": bal_test[0],
        "test_balance_mean": bal_test[1],
        "test_bal_diff_worst": bal_test_diff[0],
        "test_bal_diff_mean": bal_test_diff[1],
        "test_silhouette": compute_silhouette_safe(
            X_test_cluster.to_numpy(dtype=float), cluster_test
        ),
    }
    return res



def run_proxies_on_dataset(strategies,
                           df,
                           df_dict,
                           dataset_name,
                           ccr=(-1, -1),
                           test_size=0.3,
                           random_state=0):
    label_col = df_dict["label"]
    sens_attrs = df_dict["sens_attrs"]
    sens_name = sens_attrs[0]

    # Train/test split (we only need features + sens)
    X = df.drop(columns=[label_col])
    y = df[label_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    # Full train/test (with sens) for FCA
    df_train = df.loc[X_train.index]
    df_test = df.loc[X_test.index]

    # Remove sensitive attributes for clustering features
    X_train_cluster = X_train.drop(columns=sens_attrs)
    X_test_cluster = X_test.drop(columns=sens_attrs)

    # Sensitive vectors (use first sens attr, consistent with SFC/PFC usage)
    sens_train = df_train[sens_name].to_numpy()
    sens_test = df_test[sens_name].to_numpy()

    # Choose k via log_means like FALCC
    k = estimate_k_log_means(X_cluster=X_train_cluster,
                             df_train=df_train,
                             df_dict=df_dict,
                             ccr=ccr)

    results = []

    if "kmeans" in strategies:
        try:
            start = time.time()
            kmeans_res = eval_kmeans(
                X_train_cluster=X_train_cluster,
                X_test_cluster=X_test_cluster,
                sens_train=sens_train,
                sens_test=sens_test,
                k=k,
                random_state=random_state,
            )
            kmeans_res.update({"dataset": dataset_name, "proxy": "kmeans", "k": k, "time": time.time()-start})
            results.append(kmeans_res)
        except Exception as e:
            print(e)
 
    if "sfc" in strategies:
        try:
            start = time.time()
            sfc_res = eval_sfc(
                X_train_cluster=X_train_cluster,
                X_test_cluster=X_test_cluster,
                sens_train=sens_train,
                sens_test=sens_test,
                k=k,
                dataset_name=dataset_name,
            )
            sfc_res.update({"dataset": dataset_name, "proxy": "sfc", "k": k, "time": time.time()-start})
            results.append(sfc_res)
        except Exception as e:
            print(e)

    if "vanilla" in strategies:
        try:
            start = time.time()
            fairlet_res = eval_fairlet(
                X_train_cluster=X_train_cluster,
                X_test_cluster=X_test_cluster,
                sens_train=sens_train,
                sens_test=sens_test,
                mode="vanilla",
                k=k,
                random_state=random_state,
            )
            fairlet_res.update({"dataset": dataset_name, "proxy": "vanilla", "k": k, "time": time.time()-start})
            results.append(fairlet_res)
        except Exception as e:
            print(e)

    if "mcf" in strategies:
        try:
            start = time.time()
            fairlet_res = eval_fairlet(
                X_train_cluster=X_train_cluster,
                X_test_cluster=X_test_cluster,
                sens_train=sens_train,
                sens_test=sens_test,
                mode="mcf",
                k=k,
                random_state=random_state,
            )
            fairlet_res.update({"dataset": dataset_name, "proxy": "mcf", "k": k, "time": time.time()-start})
            results.append(fairlet_res)
        except Exception as e:
            print(e)

    if "shfd" in strategies:
        try:
            start = time.time()
            fairlet_res = eval_fairlet(
                X_train_cluster=X_train_cluster,
                X_test_cluster=X_test_cluster,
                sens_train=sens_train,
                sens_test=sens_test,
                mode="shfd_drop",
                bal="-",
                k=k,
                random_state=random_state,
            )
            fairlet_res.update({"dataset": dataset_name, "proxy": "shfd_drop", "k": k, "time": time.time()-start})
            results.append(fairlet_res)
        except Exception as e:
            print(e)
            
    return results

/Users/nicolassig/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
`load_boston` has been removed from scikit-learn since version 1.2.

The Boston housing prices dataset has an ethical problem: as
investigated in [1], the authors of this dataset engineered a
non-invertible variable "B" assuming that racial self-segregation had a
positive impact on house prices [2]. Furthermore the goal of the
research that led to the creation of this dataset was to study the
impact of air quality but it did not give adequate demonstration of the
validity of this assumption.

The scikit-learn maintainers therefore strongly discourage the use of
this dataset unless the purpose of the code is to study and educate
about ethical issues in data science and machine learning.

In this special case, you can fetch the dataset from the original
source::

In [6]:
ds_list = ["communities", "compas", "compas_sex", "drug_consumption"
           "ACS_ID_norm", "ACS_OR_norm", "credit_card_clients", 
           "adult_data_set_race", "adult_data_set_sex"]

all_results = []
strategies = ["kmeans", "reweighing", "vanilla", "shfd_drop", "sfc"]

for ds_name in ds_list:
    print(ds_name)
    df_dict = DATA_DICT[ds_name]
    df_path = os.path.join("Datasets", f"{ds_name}.csv")
    df = pd.read_csv(df_path, index_col=0)

    for csize in [15, 20]:
        res = run_proxies_on_dataset(
            strategies=strategies,
            df=df,
            df_dict=df_dict,
            dataset_name=ds_name,
            ccr=(csize, csize),        # same semantics as in FALCC
            test_size=0.3,
            random_state=0,
        )
        all_results.extend(res)
    
        results_df = pd.DataFrame(all_results)
        results_df.to_csv("proxy_clustering.csv")



drug_consumption


/Users/nicolassig/opt/anaconda3/lib/python3.9/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


start fairlets..
656 fairlets have been identified.
start fairlets..
[SHFD] round k=10: +650 pairs, 7 left / 6 right still unmatched
[SHFD] round k=20: +6 pairs, 1 left / 0 right still unmatched
start fairlets..
656 fairlets have been identified.
start fairlets..
[SHFD] round k=10: +650 pairs, 7 left / 6 right still unmatched
[SHFD] round k=20: +6 pairs, 1 left / 0 right still unmatched
